# TensorBoard-Plots der MoE-Routing-Loss-r2fix-Reproduktion

Dieses Notebook liest alle TensorBoard-Scalar-Logs aus `moe_runs/randomsearch_new_routingloss/r2fix_reproduce` und erzeugt PDF-Abbildungen fuer den Anhang der Masterarbeit:

- einen Full-Width-Plot mit Trainings-/Validierungsgenauigkeit bei Hard-Routing links und Validierungsgenauigkeit mit Top-k-Routing rechts,
- einen Full-Width-Plot der Expertennutzung mit separaten Subplots fuer `$f_{small}$`, `$f_{mid}$` und `$f_{large}$`,
- die Upper-Bound-Verlaeufe als separate Zusatzabbildung.

Die gemeinsamen Legenden enthalten die relevanten Parameter der Durchlaeufe. Die Abbildungen werden unter `moe/plots/figures_routingloss_r2fix_reproduce` gespeichert.

In [ ]:
from pathlib import Path
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, MultipleLocator, PercentFormatter
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

plt.rcParams.update({
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "lines.linewidth": 1.0,
    "lines.markersize": 3.5,
})

# Beide Hauptabbildungen werden mit voller Textbreite eingebunden.
# Etwas zusätzliche Höhe bleibt für die gemeinsame Run-Legende unterhalb.
FIGSIZE_FULL = (6.8, 3.15)
FIGSIZE_07 = (3.4, 2.75)


In [2]:
RUN_DIR_NAME = "randomsearch_new_routingloss/r2fix_reproduce"
OUT_DIR_NAME = "figures_routingloss_r2fix_reproduce"


def finde_moe_root() -> Path:
    # Findet den moe-Ordner, egal ob das Notebook aus dem Repo-Root oder aus moe/plots gestartet wird.
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        for kandidat in (base, base / "moe", base / "single_pulse_classifier_training" / "moe"):
            if (kandidat / "moe_runs" / RUN_DIR_NAME).exists():
                return kandidat
    raise FileNotFoundError(f"moe_runs/{RUN_DIR_NAME} konnte vom aktuellen Arbeitsverzeichnis aus nicht gefunden werden.")


MOE_ROOT = finde_moe_root()
RUN_ROOT = MOE_ROOT / "moe_runs" / RUN_DIR_NAME
OUT_DIR = MOE_ROOT / "plots" / OUT_DIR_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

MOE_ROOT, RUN_ROOT, OUT_DIR

(PosixPath('/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/moe'),
 PosixPath('/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/moe/moe_runs/randomsearch_new_routingloss/r2fix_reproduce'),
 PosixPath('/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/moe/plots/figures_routingloss_r2fix_reproduce'))

In [3]:
def dekodiere_float_token(token: str) -> str:
    return token.replace("p", ".").replace("em", "e-")


def format_float(value) -> str:
    if value is None:
        return "?"
    try:
        return f"{float(value):g}"
    except (TypeError, ValueError):
        return str(value)


def ergänze_config_werte(meta: dict, run_dir: Path) -> dict:
    config_path = run_dir / "run_config.json"
    if not config_path.exists():
        return meta
    try:
        config = json.loads(config_path.read_text())
    except Exception:
        return meta

    training = config.get("training", {})
    if "weight_decay" not in meta and "weight_decay" in training:
        meta["weight_decay"] = format_float(training["weight_decay"])
    if "expert_lr" not in meta and "expert_learning_rate" in training:
        meta["expert_lr"] = format_float(training["expert_learning_rate"])
    if "budget" not in meta and "budget_loss_weight" in training:
        meta["budget"] = int(float(training["budget_loss_weight"]))
    if "routing" not in meta and "routing_loss_weight" in training:
        meta["routing"] = float(training["routing_loss_weight"])
    if "seed" not in meta and "seed" in config:
        meta["seed"] = int(config["seed"])
    return meta


def setze_label(meta: dict) -> dict:
    trial = meta.get("trial")
    seed = meta.get("seed", "?")
    wd = meta.get("weight_decay", "?")
    lr = meta.get("expert_lr", "?")
    prefix = f"Trial {trial}, Seed {seed}" if trial is not None else f"Seed {seed}"
    freeze = ", freeze" if "freeze_upper_bound_patience" in meta else ""
    meta["base_label"] = f"{prefix}, lr={lr}, wd={wd}{freeze}"
    meta["label"] = meta["base_label"]
    return meta


def parse_run_name(run_name: str) -> dict:
    muster = {
        "trial": r"trial(\d+)",
        "seed": r"seed(\d+)",
        "expert_lr": r"expertlr([^_]+)",
        "rejector_lr": r"rejectorlr([^_]+)",
        "weight_decay": r"wd([^_]+)",
        "temperatur": r"temp([^_]+)",
        "budget": r"budget(\d+)",
        "routing": r"routing(\d+(?:_\d+)?)",
        "freeze_upper_bound_patience": r"freezeupperboundpat(\d+)",
        "aux_warmup": r"auxwarmup(\d+)",
    }
    meta = {"run": run_name}
    for key, pattern in muster.items():
        match = re.search(pattern, run_name)
        if not match:
            continue
        wert = match.group(1)
        if key in {"trial", "seed", "budget", "freeze_upper_bound_patience", "aux_warmup"}:
            meta[key] = int(wert)
        elif key == "routing":
            meta[key] = float(wert.replace("_", "."))
        else:
            meta[key] = dekodiere_float_token(wert)
    return setze_label(meta)


def load_tensorboard_scalars(run_root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    zeilen = []
    run_zeilen = []

    for run_dir in sorted(p for p in run_root.iterdir() if p.is_dir()):
        tb_dir = run_dir / "tensorboard"
        meta = parse_run_name(run_dir.name)
        meta = setze_label(ergänze_config_werte(meta, run_dir))
        meta.update({"path": str(run_dir), "tensorboard_path": str(tb_dir)})

        if not tb_dir.exists():
            meta.update({"status": "TensorBoard-Ordner fehlt", "n_tags": 0, "n_points": 0})
            run_zeilen.append(meta)
            continue

        accumulator = EventAccumulator(str(tb_dir), size_guidance={"scalars": 0})
        try:
            accumulator.Reload()
        except Exception as exc:
            meta.update({"status": f"Lesefehler: {exc}", "n_tags": 0, "n_points": 0})
            run_zeilen.append(meta)
            continue

        tags = accumulator.Tags().get("scalars", [])
        n_points = 0
        for tag in tags:
            events = accumulator.Scalars(tag)
            n_points += len(events)
            for event in events:
                zeilen.append({
                    **meta,
                    "tag": tag,
                    "epoche": event.step,
                    "wall_time": event.wall_time,
                    "wert": float(event.value),
                })

        meta.update({"status": "ok" if tags else "keine Scalar-Tags", "n_tags": len(tags), "n_points": n_points})
        run_zeilen.append(meta)

    scalars = pd.DataFrame(zeilen)
    runs = pd.DataFrame(run_zeilen).sort_values(
        ["status", "seed", "routing", "weight_decay", "run"],
        na_position="last",
    )
    return scalars, runs


scalars, runs = load_tensorboard_scalars(RUN_ROOT)
print(f"Geladen: {len(scalars):,} Scalar-Punkte aus {runs.query('n_points > 0').shape[0]} Durchläufen.")
display(runs[["label", "status", "n_tags", "n_points", "run"]])

Geladen: 72,100 Scalar-Punkte aus 8 Durchläufen.


,label,status,n_tags,n_points,run
3,"Seed 42, lr=8.2848e-05, wd=1e-05",ok,99,11900,seed42_routing1_5
0,"Trial 0, Seed 42, lr=8.285e-05, wd=1e-05",ok,99,10000,joint_moe_worker0_trial0_seed42_expertlr8p285e...
4,"Seed 42, lr=8.2848e-05, wd=1e-05",ok,99,9600,seed42_routing2_5
5,"Seed 43, lr=8.2848e-05, wd=1e-05",ok,99,7000,seed43_routing2
1,"Trial 1, Seed 43, lr=1.305e-06, wd=1e-06",ok,99,5700,joint_moe_worker0_trial1_seed43_expertlr1p305e...
2,"Trial 2, Seed 44, lr=1.681e-05, wd=0",ok,99,3900,joint_moe_worker0_trial2_seed44_expertlr1p681e...
6,"Seed 44, lr=8.2848e-05, wd=1e-05",ok,99,12000,seed44_routing2
7,"Seed 45, lr=8.2848e-05, wd=1e-05",ok,99,12000,seed45_routing2


In [4]:
ERWARTETE_TAGS = [
    "train/accuracy",
    "val_hard/accuracy",
    "val_hard/usage_small",
    "val_hard/usage_mid",
    "val_hard/usage_large",
    "val_topk/accuracy",
    "val_upper_bound",
    "val_budgeted_upper_bound",
]

verfuegbare_tags = sorted(scalars["tag"].unique()) if not scalars.empty else []
fehlende_tags = [tag for tag in ERWARTETE_TAGS if tag not in verfuegbare_tags]
if fehlende_tags:
    warnings.warn("Folgende erwartete TensorBoard-Tags fehlen: " + ", ".join(fehlende_tags))

print("Verfuegbare Scalar-Tags:")
for tag in verfuegbare_tags:
    print(" -", tag)

Verfuegbare Scalar-Tags:
 - learning_rate/group_0
 - learning_rate/group_1
 - train/accuracy
 - train/budget
 - train/budget_loss_weight
 - train/budget_weight
 - train/ensemble
 - train/expert_aux
 - train/expert_aux_loss_weight
 - train/expert_aux_weight
 - train/expert_large
 - train/expert_mid
 - train/expert_small
 - train/experts_frozen
 - train/only_aux_warmup
 - train/routed
 - train/routing
 - train/routing_loss_weight
 - train/routing_r1
 - train/routing_r2
 - train/routing_r2_benefit_gap
 - train/routing_r2_score_gap
 - train/routing_weight
 - train/soft_usage_large
 - train/soft_usage_mid
 - train/soft_usage_small
 - train/topk_noise_std
 - train/total
 - train/usage_large
 - train/usage_mid
 - train/usage_small
 - val_budgeted_upper_bound
 - val_expert/aux
 - val_expert/aux_weight
 - val_expert/large
 - val_expert/large_accuracy
 - val_expert/large_loss
 - val_expert/mid
 - val_expert/mid_accuracy
 - val_expert/mid_loss
 - val_expert/small
 - val_expert/small_accuracy
 - v

In [ ]:
gueltige_runs = runs.loc[runs["n_points"] > 0].copy()
gueltige_runs = gueltige_runs.sort_values(
    ["seed", "routing", "weight_decay", "run"],
    na_position="last",
)
run_order = gueltige_runs["run"].tolist()


def format_number_de(value, precision=None):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "?"

    value = float(value)

    if precision is None:
        text = f"{value:g}"
    else:
        text = f"{value:.{precision}g}"

    text = text.replace("e-0", "e-").replace("e+0", "e+")
    return text.replace(".", ",")


def scalar_weight_values(tag: str) -> dict[str, str]:
    frame = scalars.loc[
        scalars["tag"] == tag,
        ["run", "wert"],
    ]

    labels = {}

    for run, values in frame.groupby("run")["wert"]:
        unique_values = sorted(
            {float(value) for value in values}
        )

        if len(unique_values) == 1:
            labels[run] = format_number_de(unique_values[0])
        else:
            labels[run] = "→".join(
                format_number_de(value)
                for value in unique_values
            )

    return labels


budget_weight_values = scalar_weight_values(
    "train/budget_loss_weight"
)

routing_weight_values = scalar_weight_values(
    "train/routing_loss_weight"
)


def make_run_label(row):
    run = row["run"]
    seed = row.get("seed", "?")
    trial = row.get("trial", np.nan)

    try:
        lr = format_number_de(
            float(row["expert_lr"]),
            precision=3,
        )
    except (TypeError, ValueError):
        lr = str(row.get("expert_lr", "?")).replace(".", ",")

    try:
        wd = format_number_de(
            float(row["weight_decay"]),
            precision=3,
        )
    except (TypeError, ValueError):
        wd = str(row.get("weight_decay", "?")).replace(".", ",")

    parts = []

    if pd.notna(trial):
        parts.append(f"T{int(trial)}")

    parts.extend([
        f"S{seed}",
        f"lr={lr}",
        f"wd={wd}",
        f"w_budget={budget_weight_values.get(run, '?')}",
        f"w_routing={routing_weight_values.get(run, '?')}",
    ])

    freeze_pat = row.get(
        "freeze_upper_bound_patience",
        np.nan,
    )

    if pd.notna(freeze_pat):
        parts.append(
            f"freeze={int(freeze_pat)}"
        )

    return ", ".join(parts)


gueltige_runs["label"] = gueltige_runs.apply(
    make_run_label,
    axis=1,
)

label_by_run = dict(
    zip(
        gueltige_runs["run"],
        gueltige_runs["label"],
    )
)

farben = plt.colormaps["tab20"].resampled(
    max(len(run_order), 1)
)

color_by_run = {
    run: farben(i)
    for i, run in enumerate(run_order)
}


USAGE_TAGS = {
    "val_hard/usage_small": r"$f_{\mathrm{small}}$",
    "val_hard/usage_mid": r"$f_{\mathrm{mid}}$",
    "val_hard/usage_large": r"$f_{\mathrm{large}}$",
}


TOPK_TAG_LABELS = {
    "val_topk/accuracy": "Genauigkeit",
    "val_topk/total": "Gesamt-Loss",
    "val_topk/ensemble": "Ensemble-Loss",
    "val_topk/expert_small": r"Loss $f_{\mathrm{small}}$",
    "val_topk/expert_mid": r"Loss $f_{\mathrm{mid}}$",
    "val_topk/expert_large": r"Loss $f_{\mathrm{large}}$",
    "val_topk/usage_small": r"Nutzung $f_{\mathrm{small}}$",
    "val_topk/usage_mid": r"Nutzung $f_{\mathrm{mid}}$",
    "val_topk/usage_large": r"Nutzung $f_{\mathrm{large}}$",
}


def tag_frame(tag: str) -> pd.DataFrame:
    frame = scalars.loc[
        scalars["tag"] == tag,
        ["run", "label", "epoche", "wert"],
    ].copy()

    return frame.sort_values(
        ["run", "epoche"]
    )


def ist_anteil(frame: pd.DataFrame) -> bool:
    if frame.empty:
        return False

    return frame["wert"].dropna().between(
        -0.02,
        1.02,
    ).all()


def style_axis(
    ax,
    ylabel=None,
    ylim=None,
    ytick_step=None,
    percent=False,
):
    ax.set_xlabel(
        "Epoche",
        labelpad=2,
    )

    if ylabel is not None:
        ax.set_ylabel(
            ylabel,
            labelpad=2,
        )

    if ylim is not None:
        ax.set_ylim(
            *ylim
        )

    if ytick_step is not None:
        ax.yaxis.set_major_locator(
            MultipleLocator(
                ytick_step
            )
        )

    if percent:
        ax.yaxis.set_major_formatter(
            PercentFormatter(
                xmax=1.0,
                decimals=0,
            )
        )

    ax.xaxis.set_major_locator(
        MaxNLocator(
            nbins=5,
            integer=True,
        )
    )

    ax.grid(
        axis="both",
        color="0.90",
        linewidth=0.7,
        linestyle="-",
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def speichere_abbildung(fig, dateiname: str):
    path = OUT_DIR / f"{dateiname}.pdf"
    fig.savefig(path)
    print(
        f"Gespeichert: "
        f"{path.relative_to(MOE_ROOT)}"
    )


def run_legend_handles(order):
    return [
        Line2D(
            [0],
            [0],
            color=color_by_run[run],
            linewidth=1.2,
            label=label_by_run[run],
        )
        for run in order
    ]


def add_shared_run_legend(
    fig,
    order,
    fontsize=5.0,
):
    # Acht Durchläufe -> zwei Spalten -> vier kompakte Zeilen.
    fig.legend(
        handles=run_legend_handles(order),
        loc="lower center",
        bbox_to_anchor=(0.5, 0.012),
        ncol=2,
        frameon=False,
        fontsize=fontsize,
        handlelength=0.9,
        handletextpad=0.22,
        columnspacing=0.65,
        borderaxespad=0.0,
        labelspacing=0.16,
    )


def beste_runs_nach_tag(tag: str) -> list[str]:
    frame = tag_frame(tag)
    if frame.empty:
        return run_order
    ranking = (
        frame.groupby("run", as_index=False)["wert"]
        .max()
        .sort_values("wert", ascending=False)
    )
    return ranking["run"].tolist()

## Hard-Routing und Top-k-Validierungsgenauigkeit

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=FIGSIZE_FULL,
)

ax_hard, ax_topk = axes
run_order_combined = beste_runs_nach_tag("val_hard/accuracy")

# Links: Trainings- und Validierungsgenauigkeit bei Hard-Routing.
for run in run_order_combined:
    train_frame = tag_frame("train/accuracy").loc[lambda df: df["run"] == run]
    val_frame = tag_frame("val_hard/accuracy").loc[lambda df: df["run"] == run]
    color = color_by_run[run]

    if not train_frame.empty:
        ax_hard.plot(
            train_frame["epoche"],
            train_frame["wert"],
            color=color,
            linestyle="--",
            linewidth=0.8,
            alpha=0.65,
        )

    if not val_frame.empty:
        ax_hard.plot(
            val_frame["epoche"],
            val_frame["wert"],
            color=color,
            linestyle="-",
            linewidth=0.9,
            alpha=0.85,
        )

style_axis(
    ax_hard,
    ylabel="Genauigkeit",
    ylim=(0.50, 1.00),
    ytick_step=0.10,
    percent=True,
)

hard_style_handles = [
    Line2D([0], [0], color="0.25", linestyle="--", linewidth=0.9, label="Training"),
    Line2D([0], [0], color="0.25", linestyle="-", linewidth=0.9, label="Validierung"),
]
ax_hard.legend(
    handles=hard_style_handles,
    loc="lower right",
    frameon=True,
    framealpha=0.9,
    borderpad=0.3,
    fontsize=6.2,
    handlelength=1.5,
    labelspacing=0.2,
)

# Rechts: Validierungsgenauigkeit mit Top-k-Routing.
topk_frame = tag_frame("val_topk/accuracy")

for run in run_order_combined:
    run_frame = topk_frame.loc[topk_frame["run"] == run]

    if run_frame.empty:
        continue

    ax_topk.plot(
        run_frame["epoche"],
        run_frame["wert"],
        color=color_by_run[run],
        linewidth=0.9,
        alpha=0.85,
    )

style_axis(
    ax_topk,
    ylabel="Validierungsgenauigkeit",
    ylim=(0.60, 0.90),
    ytick_step=0.05,
    percent=True,
)

add_shared_run_legend(
    fig,
    run_order_combined,
    fontsize=5.0,
)

fig.subplots_adjust(
    left=0.075,
    right=0.995,
    top=0.97,
    bottom=0.36,
    wspace=0.24,
)

speichere_abbildung(
    fig,
    "routingloss_r2fix_reproduce_train_val_topk_accuracy",
)
plt.show()


## Nutzung der Klassifikatoren bei Hard-Routing

In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=FIGSIZE_FULL,
    sharex=True,
    sharey=True,
)

run_order_usage = beste_runs_nach_tag("val_hard/accuracy")

for ax, (tag, title) in zip(axes, USAGE_TAGS.items()):
    frame = tag_frame(tag)

    for run in run_order_usage:
        run_frame = frame.loc[frame["run"] == run]

        if run_frame.empty:
            continue

        ax.plot(
            run_frame["epoche"],
            run_frame["wert"],
            color=color_by_run[run],
            linewidth=0.9,
            alpha=0.85,
        )

    ax.set_title(title, pad=3)

    style_axis(
        ax,
        ylabel="Anteil" if ax is axes[0] else None,
        ylim=(0.0, 1.0),
        ytick_step=0.20,
        percent=True,
    )

add_shared_run_legend(
    fig,
    run_order_usage,
    fontsize=5.0,
)

fig.subplots_adjust(
    left=0.075,
    right=0.995,
    top=0.91,
    bottom=0.36,
    wspace=0.20,
)

speichere_abbildung(
    fig,
    "routingloss_r2fix_reproduce_val_hard_classifier_usage",
)
plt.show()


## Top-k-Routing ist im gemeinsamen Plot enthalten

In [ ]:
# Die Top-k-Validierungsgenauigkeit ist jetzt rechts im gemeinsamen
# Full-Width-Plot routingloss_r2fix_reproduce_train_val_topk_accuracy.pdf enthalten.


## Upper-Bound-Verläufe

In [ ]:
fig, ax = plt.subplots(
    figsize=FIGSIZE_07,
)

upper_specs = [
    ("val_upper_bound", "Upper Bound", "-"),
    ("val_budgeted_upper_bound", "Budgeted Upper Bound", "--"),
]
run_order_upper = beste_runs_nach_tag("val_upper_bound")

for run in run_order_upper:
    color = color_by_run[run]

    for tag, _, linestyle in upper_specs:
        frame = tag_frame(tag).loc[lambda df: df["run"] == run]

        if frame.empty:
            continue

        ax.plot(
            frame["epoche"],
            frame["wert"],
            color=color,
            linestyle=linestyle,
            linewidth=0.9,
            alpha=0.85 if linestyle == "-" else 0.70,
        )

style_axis(
    ax,
    ylabel="Validierungsgenauigkeit",
    ylim=(0.0, 1.0),
    ytick_step=0.20,
    percent=True,
)

upper_style_handles = [
    Line2D([0], [0], color="0.25", linestyle="-", linewidth=0.9, label="Upper Bound"),
    Line2D([0], [0], color="0.25", linestyle="--", linewidth=0.9, label="Budgeted Upper Bound"),
]
ax.legend(
    handles=upper_style_handles,
    loc="lower right",
    frameon=False,
    fontsize=6.0,
    handlelength=1.5,
    labelspacing=0.2,
)

add_shared_run_legend(
    fig,
    run_order_upper,
    fontsize=4.8,
)

fig.subplots_adjust(
    left=0.14,
    right=0.98,
    top=0.97,
    bottom=0.39,
)

speichere_abbildung(
    fig,
    "routingloss_r2fix_reproduce_val_upper_bounds",
)
plt.show()
